# Covasim Graph Generation & Distance Comparison — Scaling Analysis

Questo notebook:
1. Genera **3 tipologie x 3 grafi = 9 grafi** da simulazioni covasim per **4 dimensioni** (N = 500, 1000, 1500, 2000):
   - **A - Random**: layer unico, mixing omogeneo (`pop_type='random'`)
   - **B - Hybrid standard**: 4 layer (household/school/work/community), struttura per eta
   - **C - Hybrid household-heavy**: stessa struttura di B ma contatti household dominanti
2. Calcola embedding EDRep per ogni grafo
3. Misura tutte le distanze a coppie con **4 metriche**:
   - **EmbDistance** (distanza spettrale unmatched)
   - **w2_distance** (Wasserstein sulla matrice di Gram completa)
   - **w2_distance_classes (sex)** (Wasserstein per partizione sesso: femmina / maschio)
   - **w2_distance_classes (age)** (Wasserstein per partizione eta: 0-20 / 21-40 / 41-60 / 60+)
4. Visualizza le 4 matrici di distanza 9x9 come heatmap per N=1000 (riferimento)
5. Valuta Silhouette Score, Between/Within ratio e NMI per ogni N
6. Plotta come variano **score** e **distanze medie intra/inter-tipo** al crescere di N

## 1. Setup

Importiamo le librerie necessarie e aggiungiamo la root del progetto al `sys.path` in modo da poter importare i moduli locali:
- **EDRep_main.EDRep** — embedding EDRep (passeggiate casuali + decomposizione spettrale)
- **functions.w2_distance_classes** — variante della distanza di Wasserstein su partizioni di nodi
- **covasim** — simulatore agent-based di reti di contatto epidemiologiche


In [33]:
# Uncomment to install covasim if not already available
# !pip install covasim

In [34]:
import sys
import os

# Aggiunge la root del progetto al path di Python per trovare i moduli locali
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Librerie scientifiche di base
import numpy as np
import networkx as nx          # manipolazione di grafi
import ot                      # optimal transport (distanza di Wasserstein)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd

# Simulatore di reti di contatto epidemiologiche
import covasim as cv

# Metriche di clustering e valutazione
from sklearn.metrics import silhouette_score, normalized_mutual_info_score
from sklearn.decomposition import NMF   # fattorizzazione non-negativa per il clustering
from sklearn.cluster import KMeans

# Moduli locali del progetto di tesi
from EDRep_main.EDRep import NodeEmbedding   # embedding spettrale dei nodi
from functions import w2_distance_classes     # Wasserstein per partizioni


## 2. Graph Generation

Usiamo **covasim** per generare reti di contatto realistiche. Ogni grafo rappresenta una popolazione di agenti con legami sociali estratti da layer distinti.

Tre tipologie strutturalmente diverse:

| Tipo | `pop_type` | Struttura | Caratteristica |
|------|------------|-----------|----------------|
| **A — Random** | `random` | layer unico `a` | mixing omogeneo, grado quasi-regolare |
| **B — Hybrid std** | `hybrid` | 4 layer h/s/w/c | struttura per eta, default covasim |
| **C — Hybrid hh** | `hybrid` | 4 layer h/s/w/c | household molto denso, cluster locali forti |

Per ciascuna tipologia generiamo **3 grafi** con seed diversi per valutare la variabilita intra-tipo.


In [35]:
def covasim_graph(pop_size=200, seed=0, pop_type='random', contacts=None):
    """
    Esegue una simulazione covasim e restituisce la rete di contatto come grafo NetworkX.

    Parametri
    ---------
    pop_size : numero di agenti nella popolazione
    seed     : seme per la riproducibilita della simulazione
    pop_type : 'random' (layer unico omogeneo) o 'hybrid' (4 layer, struttura per eta)
    contacts : dizionario {layer: n_contatti} oppure None per i default di covasim
    """
    # Parametri base della simulazione; contacts viene aggiunto solo se specificato
    kwargs = dict(pop_size=pop_size, rand_seed=seed, pop_type=pop_type, verbose=0)
    if contacts is not None:
        kwargs['contacts'] = contacts

    # Creiamo ed eseguiamo la simulazione (un solo step basta per estrarre la rete)
    sim = cv.Sim(**kwargs)
    sim.run()

    # In covasim 3.x si usa len(sim.people) al posto del deprecato sim.n
    n = len(sim.people)
    G = nx.Graph()
    G.add_nodes_from(range(n))

    # Ogni layer (household, school, work, community...) contiene array p1 e p2
    # con gli indici dei nodi ai due estremi di ciascun contatto
    for layer in sim.people.contacts.values():
        edges = zip(layer['p1'].tolist(), layer['p2'].tolist())
        G.add_edges_from(edges)

    # Associamo a ogni nodo gli attributi demografici estratti dalla simulazione
    for i in range(n):
        G.nodes[i]['sex'] = int(sim.people.sex[i])    # 0 = femmina, 1 = maschio
        G.nodes[i]['age'] = float(sim.people.age[i])  # eta in anni

    return G


In [ ]:
POP_SIZES  = [500, 1000, 1500, 2000]  # dimensioni di grafo da confrontare
DIM        = 32    # dimensione embedding EDRep
K          = 3     # numero di cluster attesi (uno per tipologia)
SEEDS      = [0, 1, 2]  # tre seed per tipologia

GRAPH_TYPES = [
    {
        'label':    'A-Random',
        'pop_type': 'random',
        'contacts': {'a': 20},
        'desc':     'Homogeneous random mixing, single contact layer',
    },
    {
        'label':    'B-Hybrid-std',
        'pop_type': 'hybrid',
        'contacts': None,
        'desc':     'Realistic 4-layer network (household/school/work/community)',
    },
    {
        'label':    'C-Hybrid-hh',
        'pop_type': 'hybrid',
        'contacts': {'h': 8, 's': 5, 'w': 5, 'c': 5},
        'desc':     'Household-heavy: dense local clusters, sparse other layers',
    },
]

METRIC_NAMES = ['EmbDistance', 'w2_distance', 'w2_classes (sex)', 'w2_classes (age)']
GROUP_SIZE   = len(SEEDS)   # grafi per tipologia

print(f"Dimensioni da analizzare : {POP_SIZES}")
print(f"Tipi di grafo            : {[g['label'] for g in GRAPH_TYPES]}")
print(f"Semi per tipologia       : {SEEDS}")
print(f"Totale grafi per N       : {len(GRAPH_TYPES) * len(SEEDS)}")

## 3. Node Embeddings (EDRep)

**EDRep** associa a ogni nodo un vettore in $\mathbb{R}^d$ che codifica la sua struttura locale e globale nel grafo tramite decomposizione spettrale della matrice di diffusione.

`NodeEmbedding` prende la matrice di adiacenza sparsa e restituisce la matrice $X \in \mathbb{R}^{n \times d}$, dove $n$ e il numero di nodi e $d$ (`dim`) e la dimensione dello spazio di embedding.

Lo stesso `seed` numpy viene usato per tutti i grafi per garantire la riproducibilita dei passi stocastici interni.


In [37]:
def node_embedding(G, dim=16, seed=42):
    """
    Calcola l'embedding EDRep per il grafo G.

    Parametri
    ---------
    G   : grafo NetworkX
    dim : dimensione dello spazio di embedding (numero di componenti spettrali)
    seed: seme numpy per la riproducibilita

    Restituisce
    -----------
    X : matrice (n_nodi x dim) con i vettori di embedding di ogni nodo
    """
    # Convertiamo il grafo in matrice di adiacenza sparsa CSR:
    # NodeEmbedding lavora internamente con operazioni su matrici sparse
    A = nx.to_scipy_sparse_array(G, format='csr')

    np.random.seed(seed)  # fissa la randomicita prima di ogni embedding

    # k=1: un solo ordine di diffusione (passeggiate di lunghezza 1)
    emb = NodeEmbedding(A, dim=dim, k=1, verbose=False)
    return emb.X  # matrice (n_nodi x dim)


In [ ]:
# Embedding, distanze e score vengono calcolati nel loop principale (sezione 8).

## 4. Distance Functions

Quattro metriche per confrontare coppie di grafi a partire dai loro embedding:

### EmbDistance (distanza spettrale unmatched)
Confronta gli spettri delle matrici di Gram $X^\top X$ dei due embedding. Calcola $\|\lambda_1 - \lambda_2\|_2$ dove $\lambda_i$ sono gli autovalori normalizzati per $n_i$. Non richiede lo stesso numero di nodi.

### w2_distance (Wasserstein sulla matrice di Gram)
Costruisce $S = X X^\top$, estrae il triangolo superiore come distribuzione 1D e calcola la distanza di Wasserstein-2 tra le due distribuzioni.

### w2_distance_classes (sex)
Calcola la distanza separatamente per femmine e maschi e combina i risultati come norma euclidea dei blocchi. Usa l'attributo `sex` di ogni nodo.

### w2_distance_classes (age)
Stessa logica della versione per sesso, ma con **4 fasce di eta**: 0-20, 21-40, 41-60, 60+. Cattura differenze strutturali legate alla distribuzione demografica per eta.

Le funzioni `sex_partition` e `age_partition` suddividono le righe dell'embedding in base agli attributi nodali, restituendo una lista di sottomatrici (una per classe).


In [39]:
def EmbDistance(X, Y, distance_type='unmatched'):
    """
    Distanza tra due grafi basata sui loro embedding.

    - unmatched: confronta gli spettri delle Gram normalizzate (non richiede n1==n2).
    - matched  : norma di Frobenius della differenza delle Gram (richiede n1==n2).
    """
    n1, d1 = X.shape
    n2, d2 = Y.shape

    if d1 != d2:
        raise ValueError('The embedding matrices have different dimensions')
    if distance_type not in ['unmatched', 'matched']:
        raise ValueError('distance_type must be unmatched or matched')
    if distance_type == 'matched' and n1 != n2:
        raise ValueError('Matched distance requires same number of nodes')

    if distance_type == 'matched':
        # ||X^T X - Y^T Y||_F calcolata tramite l'identita bilineare
        Mxx = X.T @ X
        Mxy = X.T @ Y
        Myy = Y.T @ Y
        return np.sqrt(np.abs(
            np.linalg.norm(Mxx)**2 + np.linalg.norm(Myy)**2 - 2*np.linalg.norm(Mxy)**2
        ))
    else:
        # Autovalori della Gram normalizzata per numero di nodi.
        # La norma L2 della differenza degli spettri misura la diversita strutturale.
        lam1 = np.linalg.eigvalsh(X.T @ X) / n1
        lam2 = np.linalg.eigvalsh(Y.T @ Y) / n2
        return np.linalg.norm(lam1 - lam2)


def w2_distance(emb_1, emb_2):
    """
    Distanza di Wasserstein-2 tra due grafi tramite le loro matrici di Gram.

    S = X X^T e la matrice di similarita tra tutti i nodi. Trattando i valori
    del triangolo superiore come distribuzione 1D, la Wasserstein misura quanto
    diversamente sono distribuite tali similarita nei due grafi.
    """
    # Matrice di Gram (n x n): S[i,j] = <x_i, x_j>
    S_1 = emb_1 @ emb_1.T
    S_2 = emb_2 @ emb_2.T

    # k=1 esclude la diagonale per non contare l'auto-similarita
    idx_1 = np.triu_indices_from(S_1, k=1)
    idx_2 = np.triu_indices_from(S_2, k=1)
    vals_S_1 = S_1[idx_1]
    vals_S_2 = S_2[idx_2]

    # Wasserstein-2 unidimensionale: radice del costo di trasporto ottimale
    wd2 = ot.wasserstein_1d(vals_S_1, vals_S_2, p=2)**(1/2)
    return wd2


def sex_partition(G, X):
    """
    Separa le righe dell'embedding X in base all'attributo sex del nodo.
    Restituisce [X_femmine, X_maschi].
    """
    sexes = np.array([G.nodes[i]['sex'] for i in range(G.number_of_nodes())])
    return [X[sexes == 0], X[sexes == 1]]


def age_partition(G, X):
    """
    Separa le righe dell'embedding X in 4 fasce di eta:
        0-20 | 21-40 | 41-60 | 60+

    Restituisce una lista di 4 sottomatrici (una per fascia).
    w2_distance_classes confrontera i blocchi corrispondenti tra due grafi.
    """
    ages = np.array([G.nodes[i]['age'] for i in range(G.number_of_nodes())])
    return [
        X[ages <= 20],                          # 0-20
        X[(ages > 20) & (ages <= 40)],          # 21-40
        X[(ages > 40) & (ages <= 60)],          # 41-60
        X[ages > 60],                           # 60+
    ]


## 5. Pairwise Distance Matrices

Calcoliamo **4 matrici di distanza 9x9**, una per ciascuna metrica. Ogni cella $(i,j)$ contiene la distanza tra il grafo $i$ e il grafo $j$.

Il loop percorre solo il triangolo superiore ($i < j$) e poi simmetrizza, evitando calcoli ridondanti. Per le due varianti di `w2_distance_classes` le partizioni dei nodi (per sesso e per eta) vengono costruite al volo a partire dagli attributi del grafo.


In [ ]:
# Le matrici di distanza vengono calcolate nel loop principale (sezione 8).

## 6. Visualization

Visualizziamo le **4 matrici di distanza come heatmap** disposte in una griglia 2x2. Una buona metrica mostrera valori bassi all'interno dei blocchi diagonali (grafi dello stesso tipo) e valori alti fuori dai blocchi (tipi diversi).

Le linee navy delimitano i tre blocchi A, B, C. La colormap va da giallo (distanza bassa) a rosso (distanza alta).


In [ ]:
# Le heatmap per N=1000 vengono generate nel loop principale (sezione 8).

In [42]:
def bw_ratio(D, labels):
    """
    Between/Within ratio: rapporto tra distanza media inter-gruppo e intra-gruppo.

    Valori > 1 indicano che i grafi dello stesso tipo sono mediamente piu vicini
    tra loro che ai grafi di tipo diverso. Valori piu alti = separazione migliore.
    """
    labels = np.array(labels)
    # Coppie con stessa etichetta -> distanze intra-gruppo (within)
    within  = [D[i, j] for i in range(len(labels))
                        for j in range(i + 1, len(labels)) if labels[i] == labels[j]]
    # Coppie con etichette diverse -> distanze inter-gruppo (between)
    between = [D[i, j] for i in range(len(labels))
                        for j in range(i + 1, len(labels)) if labels[i] != labels[j]]
    return np.mean(between) / np.mean(within)


## 7. Separation Metrics

Per valutare quale distanza separa meglio le 3 tipologie di grafo usiamo tre metriche complementari, calcolate per tutte e 4 le distanze nella sezione 8:

- **Silhouette Score** (`sklearn`, `metric='precomputed'`): per ogni grafo misura quanto e piu vicino ai grafi della sua tipologia rispetto alle altre. Range [-1, 1]; valori piu alti = separazione migliore.
- **Between/Within ratio**: rapporto tra distanza media *inter*-tipologia e distanza media *intra*-tipologia. Valori > 1 e piu alti = separazione migliore.
- **NMI** (Normalized Mutual Information): confronta le etichette predette da `ClusterNMF` con il ground truth. Range [0, 1]; NMI=1 = clustering perfetto.


## 8. Loop principale — calcolo su tutti i valori di N

Per ogni dimensione in `POP_SIZES` il loop:
1. Genera i 9 grafi e calcola gli embedding
2. Calcola le 4 matrici di distanza 9×9
3. Mostra le heatmap (solo per N=1000, come riferimento)
4. Calcola Silhouette, BW Ratio e NMI per ciascuna metrica
5. Raccoglie i risultati in `results[N]` per i plot di scaling

In [43]:
def NMF_kmeans(M, k):
    """
    Singola esecuzione di NMF + k-means sulla matrice M.

    Passaggi:
    1. Aggiunge alla diagonale la media degli elementi non-zero: rende M
       strettamente positiva e migliora la stabilita della NMF.
    2. Normalizza M per la sua media globale (scala i valori intorno a 1).
    3. Applica NMF con k componenti: M ~ W H, con H di forma (k, n).
       Le colonne di H^T (n x k) sono i descrittori latenti di ciascun punto.
    4. Applica k-means su H^T per assegnare ogni punto a un cluster.

    Restituisce le etichette predette e l'inerzia k-means (usata per la selezione).
    """
    n, _ = M.shape
    # Perturbazione diagonale: rende la matrice strettamente positiva
    Mt = M + np.eye(n) * np.mean(M[M.nonzero()])
    Mt = Mt / np.mean(Mt)  # normalizzazione per stabilita numerica

    # NMF: decompone Mt ~ W @ H; components_ e H di forma (k, n)
    Y = NMF(n_components=k, max_iter=2000).fit(Mt).components_

    # K-means su Y^T (n x k): ogni riga e il vettore latente di un punto
    kmeans = KMeans(n_clusters=k, n_init=10).fit(Y.T)
    return kmeans.labels_, np.abs(kmeans.score(Y.T))  # score = inerzia (negativa)


def ClusterNMF(M, k):
    """
    Clustering robusto tramite NMF + k-means con 21 ripetizioni indipendenti.

    NMF e k-means sono sensibili all'inizializzazione casuale: eseguiamo
    l'algoritmo 21 volte e teniamo il risultato con l'inerzia minima
    (cluster piu compatti nello spazio latente).
    """
    est_l, score = NMF_kmeans(M, k)  # prima esecuzione
    for _ in range(20):              # 20 ripetizioni aggiuntive
        est_l_, score_ = NMF_kmeans(M, k)
        if score_ < score:           # teniamo il risultato con inerzia minore
            score = score_
            est_l = est_l_
    return est_l

In [ ]:
results = {}

for pop_size in POP_SIZES:
    print(f"\n{'='*60}")
    print(f"  pop_size = {pop_size}")
    print(f"{'='*60}")

    # ── 1. Generazione grafi ──────────────────────────────────────
    graphs       = []
    graph_labels = []
    type_labels  = []

    for t_idx, gtype in enumerate(GRAPH_TYPES):
        for seed in SEEDS:
            G = covasim_graph(pop_size=pop_size, seed=seed,
                              pop_type=gtype['pop_type'],
                              contacts=gtype['contacts'])
            graphs.append(G)
            graph_labels.append(f"{gtype['label'][:1]}-{seed}")
            type_labels.append(t_idx)

    N_graphs    = len(graphs)
    true_labels = np.array(type_labels)

    # ── 2. Embedding ──────────────────────────────────────────────
    embeddings = [node_embedding(G, dim=DIM) for G in graphs]

    # ── 3. Matrici di distanza ────────────────────────────────────
    D_emb = np.zeros((N_graphs, N_graphs))
    D_w2  = np.zeros((N_graphs, N_graphs))
    D_cls = np.zeros((N_graphs, N_graphs))
    D_age = np.zeros((N_graphs, N_graphs))

    for i in range(N_graphs):
        for j in range(i + 1, N_graphs):
            D_emb[i, j] = EmbDistance(embeddings[i], embeddings[j])
            D_w2 [i, j] = w2_distance(embeddings[i], embeddings[j])
            D_cls[i, j] = w2_distance_classes(
                sex_partition(graphs[i], embeddings[i]),
                sex_partition(graphs[j], embeddings[j])
            )[0]
            D_age[i, j] = w2_distance_classes(
                age_partition(graphs[i], embeddings[i]),
                age_partition(graphs[j], embeddings[j])
            )[0]
            D_emb[j,i]=D_emb[i,j]; D_w2[j,i]=D_w2[i,j]
            D_cls[j,i]=D_cls[i,j]; D_age[j,i]=D_age[i,j]

    all_dist = list(zip(METRIC_NAMES, [D_emb, D_w2, D_cls, D_age]))

    # ── 4. Heatmap 2x2 (solo per N=1000) ─────────────────────────
    if pop_size == 1000:
        type_names = [gt['label'] for gt in GRAPH_TYPES]
        fig, axes = plt.subplots(2, 2, figsize=(18, 14))
        axes = axes.flatten()
        for ax, (title, mat) in zip(axes, all_dist):
            sns.heatmap(mat, ax=ax, annot=True, fmt='.4f', cmap='YlOrRd',
                        linewidths=0.3,
                        xticklabels=graph_labels,
                        yticklabels=graph_labels)
            for k in range(GROUP_SIZE, N_graphs, GROUP_SIZE):
                ax.axhline(k, color='navy', linewidth=2)
                ax.axvline(k, color='navy', linewidth=2)
            ax.set_title(title, pad=12, fontsize=11)
            ax.tick_params(axis='x', rotation=45)
            ax.tick_params(axis='y', rotation=0)
        patches = [mpatches.Patch(label=f'Type {t}') for t in type_names]
        fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=10,
                   title='Graph types (separated by navy lines)',
                   bbox_to_anchor=(0.5, 0.01))
        plt.suptitle(
            f'Pairwise graph distances – covasim (N={pop_size}, 3 types × {GROUP_SIZE} graphs)',
            fontsize=14, y=1.01)
        plt.tight_layout()
        plt.show()

    # ── 5. Score e distanze medie ─────────────────────────────────
    scores     = {}
    mean_intra = {}
    mean_inter = {}

    for name, D in all_dist:
        sil  = silhouette_score(D, true_labels, metric='precomputed')
        bwr  = bw_ratio(D, true_labels)
        pred = ClusterNMF(D, k=K)
        nmi  = normalized_mutual_info_score(true_labels, pred)
        scores[name] = {'sil': round(sil, 4), 'bwr': round(bwr, 4), 'nmi': round(nmi, 4)}

        tl    = np.array(true_labels)
        intra = [D[i,j] for i in range(N_graphs)
                         for j in range(i+1, N_graphs) if tl[i] == tl[j]]
        inter = [D[i,j] for i in range(N_graphs)
                         for j in range(i+1, N_graphs) if tl[i] != tl[j]]
        mean_intra[name] = np.mean(intra)
        mean_inter[name] = np.mean(inter)

        print(f"  {name:25s}  sil={sil:+.4f}  bwr={bwr:.4f}  NMI={nmi:.4f}")

    results[pop_size] = {
        'scores':     scores,
        'mean_intra': mean_intra,
        'mean_inter': mean_inter,
    }

    df = pd.DataFrame(scores).T
    df.columns = ['Silhouette', 'BW Ratio', 'NMI']
    print(f"\n  Tabella N={pop_size}:\n{df.to_string()}")

print(f"\n\nRisultati disponibili per N = {list(results.keys())}")

## 9. Analisi dello scaling — come variano score e distanze con N

Plottiamo i risultati accumulati nel `results` dict per capire se le metriche di distanza diventano più o meno discriminative al crescere della dimensione del grafo.

### 9a. Score (Silhouette, BW Ratio, NMI) al variare di N

Un subplot per ogni score, una linea per metrica. Valori stabili o crescenti indicano che la metrica scala bene con N.

In [ ]:
COLORS       = ['steelblue', 'darkorange', 'seagreen', 'crimson']
SCORE_KEYS   = ['sil',        'bwr',       'nmi']
SCORE_LABELS = ['Silhouette', 'BW Ratio',  'NMI']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, sk, sl in zip(axes, SCORE_KEYS, SCORE_LABELS):
    for mname, color in zip(METRIC_NAMES, COLORS):
        vals = [results[N]['scores'][mname][sk] for N in POP_SIZES]
        ax.plot(POP_SIZES, vals, marker='o', label=mname, color=color, linewidth=2)
    ax.set_title(sl, fontsize=12)
    ax.set_xlabel('N (nodi)')
    ax.set_ylabel(sl)
    ax.set_xticks(POP_SIZES)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Score delle metriche di distanza al variare di N', fontsize=14)
plt.tight_layout()
plt.show()

### 9b. Distanze medie intra/inter-tipo al variare di N

Per ogni metrica mostriamo come crescono le distanze assolute **dentro** i cluster (intra-tipo) e **tra** cluster diversi (inter-tipo) all'aumentare di N. Un buon discriminatore mantiene un divario ampio tra le due curve.

In [ ]:
COLORS = ['steelblue', 'darkorange', 'seagreen', 'crimson']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (mname, color) in zip(axes, zip(METRIC_NAMES, COLORS)):
    intra_vals = [results[N]['mean_intra'][mname] for N in POP_SIZES]
    inter_vals = [results[N]['mean_inter'][mname] for N in POP_SIZES]
    ax.plot(POP_SIZES, intra_vals, marker='o', label='media intra-tipo',
            color='steelblue', linewidth=2)
    ax.plot(POP_SIZES, inter_vals, marker='s', label='media inter-tipo',
            color='darkorange', linewidth=2)
    ax.set_title(mname, fontsize=12)
    ax.set_xlabel('N (nodi)')
    ax.set_ylabel('Distanza media')
    ax.set_xticks(POP_SIZES)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Distanze medie intra/inter-tipo al variare di N', fontsize=14)
plt.tight_layout()
plt.show()